# *this is to check the model with our fine tuned weight*

In [1]:
!nvidia-smi
import os

# Find fine-tuned weights structure
print("=== Fine-tuned weights ===")
base = "/kaggle/input/prot2chat-tropical-finetuned"
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"  {indent}{f}")

Sun Apr  5 12:56:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.system("find /kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned -type f | head -20")

/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final/adapter_model.safetensors
/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final/README (1).md
/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final/adapter_config.json
/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final/tokenizer.json
/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final/tokenizer_config.json
/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final/special_tokens_map.json
/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/adapter_final.pth


0

# **cell 2 : clone + install******

In [3]:
import os
!git clone https://github.com/wangzc1233/Prot2Chat.git
os.chdir("Prot2Chat/prot2chat")
print("Working dir:", os.getcwd())

!pip install transformers==4.36.2 -q
!pip install peft==0.7.1 -q
!pip install gradio==4.7.1 -q
!pip install biopython -q
!pip install biotite -q
!pip install einops -q
!pip install sentencepiece -q
!pip install accelerate -q
!pip install scipy -q
!pip install langchain==0.1.0 -q
!pip install langchain-community==0.0.10 -q
!pip install modelscope -q
!pip install huggingface_hub -q
!pip install pyngrok -q
print("✅ All installed")

Cloning into 'Prot2Chat'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 173 (delta 44), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 14.74 MiB | 12.71 MiB/s, done.
Resolving deltas: 100% (44/44), done.
Working dir: /kaggle/working/Prot2Chat/prot2chat
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 82.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which 

# *cell 3 : set paths***

In [4]:
import os

WEIGHTS      = "/kaggle/input/datasets/souvikmandol/prot2chat-weights/prot2chat_weights/zcw51699/prot2chat"
BASE_MODEL_PATH = f"{WEIGHTS}/base_model"
LORA_PATH    = "/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final"
ADAPTER_PATH = "/kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/adapter_final.pth"
PORT   = 7860
GPU_ID = "0,1"

for label, path in [("BASE_MODEL", BASE_MODEL_PATH),
                    ("LORA",       LORA_PATH),
                    ("ADAPTER",    ADAPTER_PATH)]:
    print(f"{'✅' if os.path.exists(path) else '❌'} {label}: {path}")

✅ BASE_MODEL: /kaggle/input/datasets/souvikmandol/prot2chat-weights/prot2chat_weights/zcw51699/prot2chat/base_model
✅ LORA: /kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/lora_final
✅ ADAPTER: /kaggle/input/datasets/souvikmandol/tropical-dataset-finetuned/adapter_final.pth


# cell 4: fix proteinMPNN

In [5]:
import shutil, os

src = f"{WEIGHTS}/ProteinMPNN"
dst = "ProteinMPNN"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f"✅ ProteinMPNN ready at {os.path.abspath(dst)}")

✅ ProteinMPNN ready at /kaggle/working/Prot2Chat/prot2chat/ProteinMPNN


# *Cell 5 — Fix preprocess.py*

In [6]:
import os

correct_mpnn_path = os.path.abspath("ProteinMPNN")
with open("preprocess.py", "r") as f:
    lines = f.readlines()

new_lines = []
dir_path_written = False
for line in lines:
    if line.strip().startswith("# model_") and "load_model" in line:
        fixed = line.strip().lstrip("# ")
        fixed = fixed.replace("'ProteinMPNN/", f"'{correct_mpnn_path}/")
        new_lines.append(fixed + "\n")
    elif "dir_path" in line and "=" in line:
        if not dir_path_written:
            new_lines.append(f"dir_path = '{correct_mpnn_path}/'\n")
            dir_path_written = True
    elif "/data/zcwang" in line:
        line = line.replace("/data/zcwang/Protein/project/ProteinMPNN/",
                            correct_mpnn_path + "/")
        new_lines.append(line)
    else:
        new_lines.append(line)

with open("preprocess.py", "w") as f:
    f.writelines(new_lines)
print("✅ preprocess.py fixed")

✅ preprocess.py fixed


# *Cell 6 — Patch demo.py*

In [7]:
import os, re

with open("demo.py", "r") as f:
    src = f.read()

correct_mpnn_path = os.path.abspath("ProteinMPNN")

# Fix hardcoded paths
src = src.replace("/data/zcwang/Protein/project/ProteinMPNN/", correct_mpnn_path + "/")
src = src.replace("/data/zcwang/Protein/project/ProteinMPNN", correct_mpnn_path)

# Flask host
src = src.replace(
    "app.run(host='localhost', port=args.port, debug=False)",
    "app.run(host='0.0.0.0', port=args.port, debug=False)"
)

# float16
src = src.replace("torch_dtype=torch.float32", "torch_dtype=torch.float16")

# dual GPU
src = src.replace(
    'device_map="auto"',
    'device_map="auto", max_memory={0: "12GiB", 1: "12GiB"}'
)

# low_cpu_mem_usage
src = src.replace(
    '''    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto", max_memory={0: "12GiB", 1: "12GiB"}
    )''',
    '''    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto", max_memory={0: "12GiB", 1: "12GiB"},
        low_cpu_mem_usage=True
    )'''
)

# offload_folder for LoRA
src = src.replace(
    'model = PeftModel.from_pretrained(model, model_id=lora_path, config=config)',
    'model = PeftModel.from_pretrained(model, model_id=lora_path, config=config, is_trainable=False, offload_folder="offload")'
)

# adapter float16
src = src.replace(
    'adapter.eval()',
    'adapter.eval()\n    adapter = adapter.half()'
)

# float16 at inference
src = src.replace(
    'protein_embedding = adapter(protein_vector, question_hidden_state)',
    'protein_embedding = adapter(protein_vector.half(), question_hidden_state.half())'
)

with open("demo.py", "w") as f:
    f.write(src)

os.makedirs("offload", exist_ok=True)

result = __import__('subprocess').run(
    ["python", "-m", "py_compile", "demo.py"],
    capture_output=True, text=True
)
print("✅ demo.py syntax valid" if result.returncode == 0 else "❌ " + result.stderr)

✅ demo.py syntax valid


# *Cell 7 — Launch 🚀*

In [8]:
import subprocess, time
from pyngrok import ngrok, conf

NGROK_TOKEN = "3AOxyflttUu6X9069WwbLILo7HB_5Y5vCf3HyTZYje9xLac3d"
conf.get_default().auth_token = NGROK_TOKEN

cmd = [
    "python", "demo.py",
    "--model_path", BASE_MODEL_PATH,
    "--lora_path",  LORA_PATH,
    "--adapter_path", ADAPTER_PATH,
    "--port", str(PORT),
    "--gpu", GPU_ID
]

print("🚀 Starting Prot2Chat-Tropical...")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(2400):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    print(line, end="")
    if " * Running on" in line or "Serving Flask" in line:
        public_url = ngrok.connect(PORT)
        print(f"\n✅ Public URL: {public_url}")
        print("🌐 Open this URL and test with a Dengue question!")
        break
    if "Error" in line and not any(w in line for w in ["Warning", "FutureWarning", "UserWarning"]):
        print(f"\n❌ Error: {line}")
        break

for _ in range(9999):
    line = proc.stdout.readline()
    if line:
        print(line, end="")
    else:
        time.sleep(0.5)

🚀 Starting Prot2Chat-Tropical...
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-04-05 13:12:21.955850: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775394742.162777     217 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775394742.211952     217 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775394742.658874     217 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same t

KeyboardInterrupt: 